In [ ]:
pip install gradio transformers torch PyPDF2 gtts speechrecognition numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
import gradio as gr
from transformers import pipeline
import torch
import re
import os
from PyPDF2 import PdfReader
from gtts import gTTS
import tempfile
import warnings
import time
import speech_recognition as sr
import numpy as np

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning, module="gtts")

# Initialize NLP models
nlp = pipeline("text-generation", model="distilgpt2", tokenizer="distilgpt2", device=0 if torch.cuda.is_available() else -1)
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Speech recognizer setup
r = sr.Recognizer()

# Extract text from PDF resume
def extract_text_from_pdf(pdf_file):
    try:
        reader = PdfReader(pdf_file.name)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text if text else "No text found in the PDF."
    except Exception as e:
        return f"Error reading PDF: {str(e)}"

# Analyze resume and generate questions
def analyze_resume(resume_text, difficulty=1):
    generic_questions = [
        "What’s your greatest strength?",
        "Describe a challenge you overcame.",
        "Why do you want this role?"
    ]
    if not resume_text:
        return generic_questions[:difficulty]

    questions = []
    skills = re.findall(r"Skills:\s*(.*?)(?:\n|$)", resume_text, re.DOTALL | re.IGNORECASE)
    experience = re.findall(r"Experience:\s*(.*?)(?:\n[A-Z]|\Z)", resume_text, re.DOTALL | re.IGNORECASE)
    education = re.findall(r"Education:\s*(.*?)(?:\n|$)", resume_text, re.DOTALL | re.IGNORECASE)

    if skills:
        first_skill = skills[0].split(',')[0].strip()
        questions.append(f"Tell me about a time you used {first_skill} in a project.")
    if experience:
        try:
            company_name = re.search(r"at\s+([\w\s]+?)\s*\(", experience[0]) or "the company"
            if isinstance(company_name, str):
                company_name = company_name
            else:
                company_name = company_name.group(1).strip()
            questions.append(f"Can you describe a key contribution you made at {company_name}?")
        except Exception:
            pass
    if education:
        first_education = education[0].split('(')[0].strip()
        questions.append(f"How did your education at {first_education} prepare you for this role?")

    return (questions + generic_questions)[:max(1, difficulty)]

# Convert text question to speech (agent speaking)
def text_to_speech(text, output_path):
    try:
        tts = gTTS(text=text, lang="en", slow=False)
        tts.save(output_path)
        return output_path
    except Exception as e:
        return f"Error generating speech: {str(e)}"

# Real-time feedback with sentiment analysis
def provide_feedback(response):
    if not response:
        return "Please provide an answer."
    word_count = len(response.split())
    sentiment = sentiment_analyzer(response)[0]
    feedback = []
    if word_count < 20:
        feedback.append("Your answer is short. Please elaborate.")
    if "I don’t know" in response.lower():
        feedback.append("Try sharing a related experience instead.")
    if sentiment["label"] == "NEGATIVE":
        feedback.append("Try to sound more positive and confident!")
    return " ".join(feedback) or "Great answer! Well detailed and positive."

# Transcribe live audio from webcam microphone (local only)
def transcribe_audio():
    try:
        with sr.Microphone() as source:
            print("Listening... Speak now!")
            audio = r.listen(source, timeout=10, phrase_time_limit=30)
        return r.recognize_google(audio)
    except Exception as e:
        return f"Error transcribing: {str(e)}"

# Main interview function
def run_interview(mode, pdf_file, webcam_input, text_input, question_index, questions_state, responses_state, timer_state, difficulty):
    try:
        # Initialize questions if not set
        if not questions_state:
            resume_text = extract_text_from_pdf(pdf_file) if pdf_file else ""
            questions_state = analyze_resume(resume_text, difficulty)
            responses_state = [""] * len(questions_state)
            timer_state = 60

        # Process response based on mode
        user_response = None
        webcam_output = webcam_input  # Display webcam feed if provided
        if mode == "voice" and not text_input:  # Voice mode with webcam mic
            user_response = transcribe_audio()
        elif mode in ["text", "voice"] and text_input:  # Text mode or text override
            user_response = text_input
            webcam_output = None  # No webcam in text mode

        # Save response
        if user_response and 0 <= question_index < len(questions_state):
            responses_state[question_index] = user_response

        # Check if interview is complete
        if question_index >= len(questions_state):
            return "Interview complete!", "Thank you!", webcam_output, questions_state, responses_state, question_index, 0, None

        # Current question and feedback
        current_question = questions_state[question_index]
        feedback = provide_feedback(user_response) if user_response else "Please provide a response."

        # Convert question to speech for voice mode
        agent_audio = None
        if mode == "voice":
            agent_audio_path = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
            text_to_speech(current_question, agent_audio_path)
            agent_audio = agent_audio_path

        # Update timer
        timer_state = max(0, timer_state - 10)

        return current_question, feedback, webcam_output, questions_state, responses_state, question_index + 1, timer_state, agent_audio

    except Exception as e:
        return f"Error: {str(e)}", "Something went wrong.", None, [], [], 0, 60, None

# Gradio interface
with gr.Blocks(title="NOVARA AI - Real-Time Mock Interview Simulator") as demo:
    gr.Markdown("# Nancy AI - Real-Time Mock Interview Simulator")
    gr.Markdown("Practice with text or voice-based interviews using your webcam (voice mode). Questions are based on your resume.")

    question_state = gr.State(value=0)
    questions_state = gr.State(value=[])
    responses_state = gr.State(value=[])
    timer_state = gr.State(value=60)

    with gr.Row():
        pdf_input = gr.File(label="Upload PDF Resume", file_types=[".pdf"])
        difficulty = gr.Slider(1, 5, step=1, label="Difficulty Level", value=1)
        mode_input = gr.Radio(["text", "voice"], label="Interview Mode", value="text")

    with gr.Row():
        webcam_input = gr.Video(label="Your Webcam (Voice Mode)", interactive=True, visible=False)
        text_input = gr.Textbox(label="Your Response (Text Mode or Voice Override)", placeholder="Type your answer here...")

    with gr.Row():
        agent_output = gr.Audio(label="Agent's Question (Voice Mode)", interactive=False, autoplay=True, visible=False)
        question_output = gr.Textbox(label="Current Question", interactive=False)
        feedback_output = gr.Textbox(label="Real-Time Feedback", interactive=False)
        timer_display = gr.Textbox(label="Time Left (seconds)", interactive=False, value="60")

    submit_btn = gr.Button("Submit Response & Next Question")

    # Dynamic visibility based on mode
    def update_visibility(mode):
        return gr.update(visible=mode == "voice"), gr.update(visible=mode == "voice")

    mode_input.change(
        fn=update_visibility,
        inputs=mode_input,
        outputs=[webcam_input, agent_output]
    )

    submit_btn.click(
        fn=run_interview,
        inputs=[mode_input, pdf_input, webcam_input, text_input, question_state, questions_state, responses_state, timer_state, difficulty],
        outputs=[question_output, feedback_output, webcam_input, questions_state, responses_state, question_state, timer_state, agent_output]
    )

demo.launch()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0081c91e4ad953ccf7.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
